# DeepSeek Free — Google Colab demo

**Read this first:** Colab's free T4 has 16 GB VRAM. DeepSeek-V4.1-Flash is a 552B-parameter model (~259 GB at IQ1_M, ~414 GB at Q4). It **cannot** run on Colab.

This notebook runs `deepseek-r1:8b` instead, which fits comfortably. For lifetime hosting use the Oracle Always Free VM + `scripts/setup.sh` path in the README.

Colab limits: ~12 h continuous, disconnects when idle, and wipes the VM afterwards. Demo only.

## 1. Install Ollama and start the server

In [ ]:
import subprocess, time, os, urllib.request

subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.Popen('nohup ollama serve > /tmp/ollama.log 2>&1 &', shell=True)

for i in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        print('Ollama is up')
        break
    except Exception:
        time.sleep(1)
else:
    print(open('/tmp/ollama.log').read()[-2000:])

## 2. Pull a model that fits a T4

In [ ]:
# 8b Q4 needs ~5-6 GB VRAM. Use deepseek-r1:1.5b if you hit OOM.
!ollama pull deepseek-r1:8b
!ollama list

## 3. Test the API

In [ ]:
import requests

r = requests.post(
    'http://127.0.0.1:11434/v1/chat/completions',
    json={
        'model': 'deepseek-r1:8b',
        'messages': [{'role': 'user', 'content': 'Say hello in one short sentence.'}],
        'max_tokens': 64,
    },
    timeout=300,
)
print(r.status_code)
print(r.text[:2000])

## 4. Optional: temporary public URL

The URL changes on every restart. Never leave an unauthenticated Ollama API public for long.

In [ ]:
import platform, subprocess, re, time

arch = 'arm64' if platform.machine() in ('aarch64', 'arm64') else 'amd64'
subprocess.run(
    f'curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch} -o /usr/local/bin/cloudflared',
    shell=True, check=True,
)
subprocess.run('chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
subprocess.Popen('nohup cloudflared tunnel --url http://127.0.0.1:11434 > /tmp/tunnel.log 2>&1 &', shell=True)

for _ in range(30):
    log = open('/tmp/tunnel.log').read()
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', log)
    if m:
        print('Tunnel:', m.group(0))
        break
    time.sleep(1)
else:
    print(open('/tmp/tunnel.log').read()[-2000:])

## Lifetime alternative

Oracle Cloud Always Free (Ampere ARM, 24 GB RAM) runs `deepseek-r1:8b` 24/7 at no cost. Clone the repo there and run `./scripts/setup.sh`.